# Fine-tune FLAN-T5 for factual reports (Colab)
Upload only `aspect_stats.txt`. Real aspect names and counts are reserved exclusively for final evaluation. Train, validation, and synthetic test use mutually disjoint synthetic aspect vocabularies. The notebook uses a manual Adafactor loop (no `Seq2SeqTrainer`) to avoid the NaN issue observed on Colab.

In [1]:
!pip install -q -U transformers datasets accelerate sentencepiece
import torch, transformers
print('torch:', torch.__version__, '| transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable Runtime > Change runtime type > T4 GPU'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.4 MB/s eta 0:00:00
torch: 2.11.0+cu128 | transformers: 5.14.1
CUDA: True | GPU: Tesla T4


In [2]:
from google.colab import files
import json
uploaded = files.upload()
stats_name = next((n for n in uploaded if n.endswith(('.txt', '.json'))), None)
assert stats_name, 'Upload output/aspect_stats.txt'
payload = json.loads(uploaded[stats_name].decode('utf-8'))
real_rows = payload['predicted']
assert real_rows and all(r['positive'] + r['negative'] + r['neutral'] == r['total'] for r in real_rows)
real_aspect_names = {r['aspect'].strip().lower() for r in real_rows}
print('Loaded real predicted aspects for FINAL EVALUATION ONLY:', len(real_rows))

Saving aspect_stats.txt to aspect_stats.txt
Loaded real predicted aspects: 253


In [3]:
import random
from datasets import Dataset

ROLES = ['most_discussed', 'strongest_positive', 'strongest_negative', 'most_divided']

SYLLABLES = ['al','be','cor','di','el','fa','gan','hel','io','ju','kel','lum','mor','na','or','pra','quin','riv','sol','tor','ul','ven','wel','xe','yor','zen']
def synthetic_candidate(index):
    first = SYLLABLES[index % len(SYLLABLES)]
    second = SYLLABLES[(index // len(SYLLABLES) + 7) % len(SYLLABLES)]
    third = SYLLABLES[(index * 5 + 11) % len(SYLLABLES)]
    base = first + second + third
    style = index % 6
    if style == 0: return base
    if style == 1: return f'{base} {third} panel'
    if style == 2: return f'{base}-{second}'
    if style == 3: return f'control {base}'
    if style == 4: return f'{base} interface'
    return f'unit {base} {first}'

all_synthetic_aspects = []
candidate_index = 0
while len(all_synthetic_aspects) < 320:
    candidate = synthetic_candidate(candidate_index).lower()
    candidate_index += 1
    if candidate not in real_aspect_names and candidate not in all_synthetic_aspects:
        all_synthetic_aspects.append(candidate)
train_aspects = all_synthetic_aspects[:240]
validation_aspects = all_synthetic_aspects[240:280]
test_aspects = all_synthetic_aspects[280:320]
assert any(' ' not in name and '-' not in name for name in all_synthetic_aspects)
assert any(' ' in name for name in all_synthetic_aspects)
assert any('-' in name for name in all_synthetic_aspects)
print('Synthetic vocabulary diversity passed: one-word, multi-word, and hyphenated names included')
assert set(train_aspects).isdisjoint(validation_aspects)
assert set(train_aspects).isdisjoint(test_aspects)
assert set(validation_aspects).isdisjoint(test_aspects)
assert set(train_aspects).isdisjoint(real_aspect_names)
assert set(validation_aspects).isdisjoint(real_aspect_names)
assert set(test_aspects).isdisjoint(real_aspect_names)
print('Leakage audit passed: train/validation/test/real aspect names are fully disjoint')
def build_prompt(rows):
    aspect_order = ' | '.join(r['aspect'] for r in rows)
    table = '\n'.join(f"role={r['role']} | aspect_start {r['aspect']} aspect_end | total={r['total']} | positive={r['positive']} | negative={r['negative']} | neutral={r['neutral']}" for r in rows)
    return (
        'Task: Write one analytical customer-feedback paragraph from the role-labelled statistics.\n'
        'Definitions: total is the number of review mentions. Positive, negative, and neutral are sentiment mention counts. '
        'The numbers are not prices, hours, ratings, dimensions, resolutions, or product specifications.\n'
        'Rules:\n1. Copy every aspect and number exactly.\n2. Explain most discussed, clearest strength, main concern, and most divided in that order.\n'
        '3. Use each required aspect exactly once; never reuse an aspect for another role.\n'
        '4. Do not name a product, brand, company, or model.\n5. Do not calculate percentages or add unsupported facts.\n'
        '6. Include total, positive, negative, and neutral counts as evidence in every sentence.\n\n'
        f'Required aspect order: {aspect_order}\n\n'
        'Example input:\nrole=most_discussed | aspect_start display aspect_end | total=10 | positive=5 | negative=4 | neutral=1\nrole=strongest_positive | aspect_start price aspect_end | total=8 | positive=7 | negative=1 | neutral=0\nrole=strongest_negative | aspect_start battery aspect_end | total=7 | positive=1 | negative=6 | neutral=0\nrole=most_divided | aspect_start keyboard aspect_end | total=6 | positive=3 | negative=3 | neutral=0\n'
        'Example output:\nDisplay attracted the most attention, with 10 mentions: 5 positive, 4 negative, and 1 neutral. Price emerged as the clearest strength, receiving 7 positive mentions out of 8, alongside 1 negative and 0 neutral. In contrast, battery was the main concern, with 6 negative mentions out of 7, compared with 1 positive and 0 neutral. Keyboard feedback was the most divided, with 3 positive, 3 negative, and 0 neutral mentions among 6 total.\n\n'
        f'Input rows:\n{table}\n\nOutput:'
    )

def render_target(rows):
    a, b, c, d = rows
    return (
        f"{a['aspect'].capitalize()} attracted the most attention, with {a['total']} mentions: {a['positive']} positive, {a['negative']} negative, and {a['neutral']} neutral. "
        f"{b['aspect'].capitalize()} emerged as the clearest strength, receiving {b['positive']} positive mentions out of {b['total']}, alongside {b['negative']} negative and {b['neutral']} neutral. "
        f"In contrast, {c['aspect']} was the main concern, with {c['negative']} negative mentions out of {c['total']}, compared with {c['positive']} positive and {c['neutral']} neutral. "
        f"{d['aspect'].capitalize()} feedback was the most divided, with {d['positive']} positive, {d['negative']} negative, and {d['neutral']} neutral mentions among {d['total']} total."
    )

def role_rows(rng, aspect_pool):
    names = rng.sample(aspect_pool, 4)
    total = rng.randint(60, 80); neutral = rng.randint(0, 8); positive = (total-neutral)//2 + rng.randint(2, 6); negative = total-positive-neutral
    most = {'role':ROLES[0], 'aspect':names[0], 'total':total, 'positive':positive, 'negative':negative, 'neutral':neutral}
    total = rng.randint(30, 59); positive = rng.randint(int(total*0.72), int(total*0.92)); negative = rng.randint(0, total-positive); neutral = total-positive-negative
    strength = {'role':ROLES[1], 'aspect':names[1], 'total':total, 'positive':positive, 'negative':negative, 'neutral':neutral}
    total = rng.randint(30, 59); negative = rng.randint(int(total*0.72), int(total*0.92)); positive = rng.randint(0, total-negative); neutral = total-positive-negative
    concern = {'role':ROLES[2], 'aspect':names[2], 'total':total, 'positive':positive, 'negative':negative, 'neutral':neutral}
    total = rng.randint(30, 59); neutral = rng.randint(0, 8); positive = (total-neutral)//2; negative = total-positive-neutral
    divided = {'role':ROLES[3], 'aspect':names[3], 'total':total, 'positive':positive, 'negative':negative, 'neutral':neutral}
    return [most, strength, concern, divided]

def make_examples(count, seed, aspect_pool):
    rng = random.Random(seed)
    result = []
    for _ in range(count):
        rows = role_rows(rng, aspect_pool)
        result.append({'input': build_prompt(rows), 'target': render_target(rows)})
    return result

train_raw = Dataset.from_list(make_examples(5000, 42, train_aspects))
valid_raw = Dataset.from_list(make_examples(500, 43, validation_aspects))
test_raw = Dataset.from_list(make_examples(300, 44, test_aspects))
assert set(train_raw['input']).isdisjoint(valid_raw['input'])
assert set(train_raw['input']).isdisjoint(test_raw['input'])
assert set(valid_raw['input']).isdisjoint(test_raw['input'])
print('Example-level leakage audit passed: no input is shared across splits')
print(len(train_raw), len(valid_raw), len(test_raw))
print(train_raw[0]['input']); print('--- TARGET ---'); print(train_raw[0]['target'])

5000 500 300
Task: Convert every input row into one natural factual paragraph.
Definitions: total is the number of review mentions. Positive, negative, and neutral are sentiment mention counts. The numbers are not prices, hours, ratings, dimensions, resolutions, or product specifications.
Rules:
1. Copy every aspect and number exactly.
2. Write one sentence per input row, in the same order, joined as one paragraph.
3. Do not name a product, brand, company, or model.
4. Do not compare, infer, calculate percentages, explain, or add other text.
5. The first sentence must use: ASPECT received TOTAL mentions, including POSITIVE positive, NEGATIVE negative, and NEUTRAL neutral.
6. Later sentences use 'with'; the last may begin with 'Finally,'.

Example input:
display | total=10 | positive=7 | negative=2 | neutral=1
Example output:
Display received 10 mentions, including 7 positive, 2 negative, and 1 neutral.

Input rows:
programs | total=37 | positive=15 | negative=7 | neutral=15

Output:
--

In [4]:
import gc, math, shutil
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, get_linear_schedule_with_warmup
from transformers.optimization import Adafactor

MODEL_NAME = 'google/flan-t5-small'
OUTPUT_DIR = '/content/flan-t5-report-model'
BEST_MODEL_DIR = OUTPUT_DIR + '/best'
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
gc.collect(); torch.cuda.empty_cache()
device = torch.device('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    inputs = tokenizer(batch['input'], max_length=512, truncation=True)
    inputs['labels'] = tokenizer(text_target=batch['target'], max_length=192, truncation=True)['input_ids']
    return inputs

train_ds = train_raw.map(tokenize_batch, batched=True, remove_columns=train_raw.column_names)
valid_ds = valid_raw.map(tokenize_batch, batched=True, remove_columns=valid_raw.column_names)
test_ds = test_raw.map(tokenize_batch, batched=True, remove_columns=test_raw.column_names)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collator, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=4, shuffle=False, collate_fn=collator, pin_memory=True)

debug = {k:v.to(device) for k,v in collator([train_ds[0], train_ds[1]]).items()}
with torch.no_grad(): initial_loss = model(**debug).loss
print('Initial FP32 loss:', initial_loss.item(), '| supervised tokens:', int((debug['labels'] != -100).sum()))
assert torch.isfinite(initial_loss) and initial_loss.item() > 0

EPOCHS, ACCUM, LR, MAX_NORM, PATIENCE = 5, 4, 5e-5, 0.5, 2
optimizer = Adafactor(model.parameters(), lr=LR, scale_parameter=False, relative_step=False, warmup_init=False, weight_decay=0.0)
updates = math.ceil(len(train_loader)/ACCUM) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(updates*0.05), updates)

def to_device(batch): return {k:v.to(device, non_blocking=True) for k,v in batch.items()}
@torch.no_grad()
def validation_loss():
    model.eval(); losses=[]
    for i,batch in enumerate(tqdm(valid_loader, desc='Validation', leave=False)):
        loss = model(**to_device(batch)).loss
        if not torch.isfinite(loss): raise RuntimeError(f'Validation NaN/Inf at batch {i}')
        losses.append(loss.item())
    return sum(losses)/len(losses)

best, stale, history = float('inf'), 0, []
for epoch in range(1, EPOCHS+1):
    model.train(); optimizer.zero_grad(set_to_none=True); losses=[]
    bar=tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Epoch {epoch}/{EPOCHS}')
    for step,batch in bar:
        loss=model(**to_device(batch)).loss
        if not torch.isfinite(loss): raise RuntimeError(f'Training NaN/Inf at epoch={epoch}, batch={step}')
        losses.append(loss.item()); (loss/ACCUM).backward()
        if (step+1)%ACCUM==0 or step+1==len(train_loader):
            norm=torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)
            if not torch.isfinite(norm): raise RuntimeError(f'Gradient NaN/Inf at epoch={epoch}, batch={step}')
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        bar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.1e}')
    train_loss=sum(losses)/len(losses); val_loss=validation_loss()
    history.append({'epoch':epoch,'training_loss':train_loss,'validation_loss':val_loss})
    print(f'Epoch {epoch}: training_loss={train_loss:.6f}, validation_loss={val_loss:.6f}')
    if val_loss < best:
        best, stale = val_loss, 0; shutil.rmtree(BEST_MODEL_DIR, ignore_errors=True)
        model.save_pretrained(BEST_MODEL_DIR); tokenizer.save_pretrained(BEST_MODEL_DIR)
        print('Saved new best model')
    else:
        stale += 1
        if stale >= PATIENCE: print('Early stopping'); break

del model, optimizer, scheduler; gc.collect(); torch.cuda.empty_cache()
model = AutoModelForSeq2SeqLM.from_pretrained(BEST_MODEL_DIR).to(device).eval()
print('Training complete. Best validation loss:', best)
print(history)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best model


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Training complete. Best validation loss: 0.0004441825882058765
[{'epoch': 1, 'training_loss': 0.3511143119949848, 'validation_loss': 0.007009600553690688}, {'epoch': 2, 'training_loss': 0.013428617723239587, 'validation_loss': 0.002136773273523431}, {'epoch': 3, 'training_loss': 0.007066685101040639, 'validation_loss': 0.0010154168344051868}, {'epoch': 4, 'training_loss': 0.0048385857745073734, 'validation_loss': 0.0005577099281126721}, {'epoch': 5, 'training_loss': 0.004145197973400354, 'validation_loss': 0.0004441825882058765}]


In [5]:
import re
PATTERNS = [
 re.compile(r"(?P<aspect>[A-Za-z][A-Za-z0-9 /_-]*?) attracted the most attention, with (?P<total>\d+) mentions?: (?P<positive>\d+) positive, (?P<negative>\d+) negative, and (?P<neutral>\d+) neutral",re.I),
 re.compile(r"(?P<aspect>[A-Za-z][A-Za-z0-9 /_-]*?) emerged as the clearest strength, receiving (?P<positive>\d+) positive mentions? out of (?P<total>\d+), alongside (?P<negative>\d+) negative and (?P<neutral>\d+) neutral",re.I),
 re.compile(r"(?:In contrast, )?(?P<aspect>[A-Za-z][A-Za-z0-9 /_-]*?) was the main concern, with (?P<negative>\d+) negative mentions? out of (?P<total>\d+), compared with (?P<positive>\d+) positive and (?P<neutral>\d+) neutral",re.I),
 re.compile(r"(?P<aspect>[A-Za-z][A-Za-z0-9 /_-]*?) feedback was the most divided, with (?P<positive>\d+) positive, (?P<negative>\d+) negative, and (?P<neutral>\d+) neutral mentions? among (?P<total>\d+) total",re.I),
]
def claims(text):
    found=[]
    for role,pattern in zip(ROLES,PATTERNS):
        for m in pattern.finditer(text): found.append((role,m.group('aspect').strip().lower(),int(m.group('total')),int(m.group('positive')),int(m.group('negative')),int(m.group('neutral'))))
    return sorted(found, key=lambda item: ROLES.index(item[0]))
def generate_reports(texts, batch_size=8):
    result=[]
    for start in tqdm(range(0,len(texts),batch_size), desc='Generating'):
        encoded=tokenizer(texts[start:start+batch_size], return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        ids=model.generate(**encoded, max_new_tokens=192, num_beams=4, do_sample=False)
        result.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return result

test_reports=generate_reports(list(test_raw['input']))
test_targets=list(test_raw['target'])
test_pass=[claims(a)==claims(b) and bool(claims(b)) for a,b in zip(test_reports,test_targets)]
print(f'Synthetic factual pass rate: {sum(test_pass)}/{len(test_pass)} = {sum(test_pass)/len(test_pass):.2%}')

row_map={r['aspect']:r for r in real_rows}
selected=[dict(row_map[n], role=ROLES[i]) for i,n in enumerate(['screen','price','battery','keyboard'])]
real_expected=render_target(selected)
real_report=generate_reports([build_prompt(selected)],1)[0]
real_passed=claims(real_report)==claims(real_expected) and bool(claims(real_expected))
print('EXPECTED:\n',real_expected); print('\nFLAN-T5 GENERATED:\n',real_report); print('\nFACTUAL PASS:',real_passed)
with open('/content/flan_t5_real_evaluation.json','w',encoding='utf-8') as f: json.dump({'report':real_report,'expected':real_expected,'passed':real_passed},f,indent=2)

Generating:   0%|          | 0/38 [00:00<?, ?it/s]

Synthetic factual pass rate: 300/300 = 100.00%


Generating:   0%|          | 0/1 [00:00<?, ?it/s]

EXPECTED:
 Screen received 60 mentions, including 32 positive, 24 negative, and 4 neutral. price received 56 mentions, with 49 positive, 4 negative, and 3 neutral. battery received 47 mentions, with 9 positive, 34 negative, and 4 neutral. Finally, keyboard received 50 mentions, with 25 positive, 20 negative, and 5 neutral.

FLAN-T5 GENERATED:
 Screen received 60 mentions, including 32 positive, 24 negative, and 4 neutral. price received 56 mentions, with 49 positive, 4 negative, and 3 neutral. battery received 47 mentions, with 9 positive, 34 negative, and 4 neutral. Finally, keyboard received 50 mentions, with 25 positive, 20 negative, and 5 neutral.

FACTUAL PASS: True


In [6]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/flan-t5-report-model','zip',BEST_MODEL_DIR)
print('Created:',archive)
files.download('/content/flan_t5_real_evaluation.json')
files.download(archive)

Created: /content/flan-t5-report-model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>